# **Claude Code Training — Module 3:** <a name="module-3"></a>
## NGS Alignment Pipeline

**Paired with:** Sessions 6–7 (bowtie2, samtools, SRA)
**Lab context:** *E. coli* K-12 MG1655 | bowtie2 | samtools | fastq-dump | makegff.py

---

**Contents:**

**Question 1: How bowtie2 works ([Go link](#issue-3_1))**

**Question 2: samtools — SAM vs BAM, sort, index ([Go link](#issue-3_2))**

**Question 3: Plan mode for pipelines ([Go link](#issue-3_3))**

**Question 4: SRA data download ([Go link](#issue-3_4))**

**Question 5: SAM FLAG values ([Go link](#issue-3_5))**

**Question 6: Full pipeline on sample data ([Go link](#issue-3_6))**

### **# Question 3-1** <a name="issue-3_1"></a>

`bowtie2` aligns short reads to a reference genome using three key mechanisms. First, it builds an **FM-index** from the reference genome — a compressed, full-text index based on the Burrows-Wheeler Transform that enables fast substring lookup without scanning the entire genome for every read. Second, it uses a **seed-and-extend** strategy: short k-mer seeds from the read are looked up in the FM-index to locate candidate alignment positions, then the full alignment is extended from those positions using dynamic programming. Third, unlike its predecessor `bowtie1`, `bowtie2` supports **gapped alignment** — it can handle insertions and deletions in the read relative to the reference, which is essential for real sequencing data where small indels are common.

Before aligning reads, you must first run `bowtie2-build` to construct the FM-index from the reference FASTA. This produces a set of index files (`.bt2` extensions) that `bowtie2` reads during alignment. Without the index, `bowtie2` cannot run — it has no data structure to search.

* Checklist:
  * **Use Claude Code to understand how bowtie2 finds alignments and what the key flags control. Write a 3-sentence summary in the answer sheet in your own words.**
  * **In the answer sheet: what does `bowtie2-build` do and why must it run before alignment?**

  **Answer sheet**
  >*Please write down your answer on the sheet below.<br/>*
  **You can use the answer writing sheet by double-clicking the relevant sheet.**
  ```
  Answer:
  ```

### **# Question 3-2** <a name="issue-3_2"></a>

**SAM** (Sequence Alignment/Map) is a plain-text, tab-delimited format — human-readable but large and slow to process. **BAM** is the binary-compressed equivalent, typically 3–5× smaller and orders of magnitude faster for random access, which is why all downstream tools expect BAM.

A BAM index (`.bai`) stores byte offsets for fixed-size genomic bins. For the index to work, all reads in a bin must be physically contiguous in the file — meaning reads must be **sorted by genomic coordinate** first. If the BAM is unsorted (reads appear in sequencing order), the index cannot map a genomic region to a unique byte range. Tools like `samtools mpileup`, IGV, and coverage calculations all require a coordinate-sorted and indexed BAM.

* Checklist:
  * **Without asking Claude Code first: write in the answer sheet why BAM must be sorted before indexing. Then verify with Claude Code — were you right?**
  * **In the code cell: write the three samtools commands to convert SAM → BAM, sort, and index.**

  **Answer sheet**
  >*Please write down your answer on the sheet below.<br/>*
  **You can use the answer writing sheet by double-clicking the relevant sheet.**
  ```
  Answer:
  ```

### **# Question 3-3** <a name="issue-3_3"></a>

Claude Code's **plan mode** lets you see every step Claude intends to take before a single command is executed. Press **Shift+Tab** before sending your prompt — the interface switches to plan mode, Claude shows you the full sequence of actions, and waits for your approval before anything runs. This is especially important for multi-step pipelines where a mistake at an early step (wrong index path, wrong flag) can silently produce incorrect output downstream.

Always use plan mode for multi-step pipelines. Review and approve before anything runs.

Target pipeline for this question:
```
FASTQ → bowtie2 align → samtools view → samtools sort → samtools index → makegff.py
```

* Checklist:
  * **Use plan mode (Shift+Tab) to generate the full pipeline. Review the proposed steps.**
  * **In the answer sheet: what did you change or verify before approving the plan? Name at least one flag or parameter.**

  **Answer sheet**
  >*Please write down your answer on the sheet below.<br/>*
  **You can use the answer writing sheet by double-clicking the relevant sheet.**
  ```
  Answer:
  ```

### **# Question 3-4** <a name="issue-3_4"></a>

`fastq-dump` (part of the SRA Toolkit) downloads sequencing data from NCBI's Sequence Read Archive by accession number (e.g., `SRR12345678`). The most important flag for paired-end data is `--split-files`. Without it, `fastq-dump` interleaves both reads into a single FASTQ file — a format most aligners do not accept as paired-end input. With `--split-files`, it writes `_1.fastq` and `_2.fastq` separately, which is what `bowtie2 -1` and `-2` expect.

Other useful flags: `--gzip` (compress output on the fly), `--outdir` (specify output directory), `--skip-technical` (drop technical reads like barcodes).

* Checklist:
  * **Generate the `fastq-dump` command for a GEO accession using Claude Code. Paste the command in the code cell with a comment explaining each flag.**
  * **In the answer sheet: what does each flag in your command do?**

  **Answer sheet**
  >*Please write down your answer on the sheet below.<br/>*
  **You can use the answer writing sheet by double-clicking the relevant sheet.**
  ```
  Answer:
  ```

### **# Question 3-5** <a name="issue-3_5"></a>

The SAM **FLAG** field is a bitwise integer encoding alignment properties. Each bit position encodes a binary property; the value you see in column 2 of a SAM record is the sum of all set bits.

Common values to know:
- `0` — read mapped to forward strand
- `16` — read mapped to reverse strand
- `4` — read is unmapped

You can filter reads by FLAG using `samtools view -f` (keep reads WITH the flag) or `-F` (exclude reads WITH the flag). You can decode any FLAG value by asking Claude Code or using the Broad Institute's flag explainer.

* Checklist:
  * **Look up what a specific SAM FLAG value means using Claude Code. Choose a value not listed above.**
  * **In the answer sheet: what FLAG value did you look up? What does it mean? How would you filter for it using samtools?**

  **Answer sheet**
  >*Please write down your answer on the sheet below.<br/>*
  **You can use the answer writing sheet by double-clicking the relevant sheet.**
  ```
  Answer:
  ```

### **# Question 3-6** <a name="issue-3_6"></a>

Run the complete NGS alignment pipeline from start to finish. Download the required files from NCBI following `data/sample/README.md`, then use plan mode (Shift+Tab) to generate and review the full pipeline before running it.

Pipeline scaffold:
```bash
# Step 1: Build bowtie2 index from reference genome
bowtie2-build <reference.fasta> <index_name>

# Step 2: Align reads
bowtie2 -x <index> -U <reads.fastq> -S output.sam

# Step 3: Convert SAM → BAM
samtools view -bS output.sam > output.bam

# Step 4: Sort
samtools sort output.bam -o output_sorted.bam

# Step 5: Index
samtools index output_sorted.bam

# Step 6: Generate GFF
python makegff.py output_sorted.bam > output.gff
```

* Checklist:
  * **Download the reference genome and reads from NCBI following `data/sample/README.md`.**
  * **Use plan mode to generate the pipeline. Paste the final commands in the code cell.**
  * **In the answer sheet: what output files did you produce? What would you load into MetaScope?**

  **Answer sheet**
  >*Please write down your answer on the sheet below.<br/>*
  **You can use the answer writing sheet by double-clicking the relevant sheet.**
  ```
  Answer:
  ```